# MicroDuck 直播 Notebook：篮球平衡 / PPO

把任务、策略、ONNX、MuJoCo 和 RDK X5/BPU 验收串成一条可复用流程。

## 0. 运行说明

建议在 `02-可运行代码/microduck-playground-stilts` 的 Python 环境中启动 Jupyter。训练模型和板端 HBM 不随 Git 提交；通过 `MICRODUCK_ONNX`、`RDK_BPU_HBM` 和 `RDK_HOST` 注入。

第一次运行请先打开 [MuJoCo 策略控制 HelloWorld](./00_MuJoCo策略控制_HelloWorld.ipynb)：自动下载行走模型，用 Python 下发前进、转向和停止命令，完整执行观测、推理、关节目标和物理更新，再进入当前任务。

板端实践通过 `MICRODUCK_RUN_BPU=1` 启用；编译通过 `MICRODUCK_COMPILE_BPU=1` 启用。工作站推理可以独立运行，关闭板端时会输出 `NOT-RUN-BPU`。

In [1]:
from pathlib import Path
import json
import os
import platform
import shlex
import socket
import subprocess


def locate_topic_root():
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / "02-可运行代码").is_dir() and (base / "01-任务资料").is_dir():
            return base
        group = base / "16-专题组队学习"
        if group.is_dir():
            matches = list(group.glob("*/02-可运行代码"))
            if matches:
                return matches[0].parent
    configured = os.getenv("MICRODUCK_TOPIC_ROOT")
    if configured and Path(configured).is_dir():
        return Path(configured).resolve()
    raise RuntimeError("找不到专题目录，请从 03-Notebook 启动 Jupyter，或设置 MICRODUCK_TOPIC_ROOT。")


TOPIC_ROOT = locate_topic_root()
PLAYGROUND_ROOT = TOPIC_ROOT / "02-可运行代码" / "microduck-playground-stilts"
OUTPUT_ROOT = TOPIC_ROOT / "03-Notebook" / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("专题目录:", TOPIC_ROOT)
print("可运行代码:", PLAYGROUND_ROOT)
print("输出目录:", OUTPUT_ROOT)
print("Jupyter 工作站:", socket.gethostname(), platform.system(), platform.machine())
print("RDK 地址:", os.getenv("RDK_HOST", "192.168.8.128"), "用户: sunrise")

专题目录: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题
可运行代码: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/02-可运行代码/microduck-playground-stilts
输出目录: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs
Jupyter 工作站: ubuntu Linux x86_64
RDK 地址: 192.168.8.128 用户: sunrise


## 1. 模型契约与本地检查

目标契约是 actor `obs[1, 61] -> action[1, 14]`。如果本任务没有随专题提交 checkpoint/ONNX，Notebook 会明确显示“模型未提供”，不会用别的任务模型冒充当前任务结果。

In [2]:
TASK_ID = 'Mjlab-Basketball-MicroDuck'
MODEL_HINT = None
import sys
sys.path.insert(0, str(TOPIC_ROOT / "03-Notebook"))
from notebook_runtime import resolve_model, policy_python, inspect_policy, local_model_check, verify_gpu_report, verify_bpu_report
TASK_KEY = 'basketball'
MODEL_PATH = resolve_model(TASK_KEY, TOPIC_ROOT, MODEL_HINT)
POLICY_PYTHON = policy_python(PLAYGROUND_ROOT)
print("任务:", TASK_KEY)
print("实际推理模型:", MODEL_PATH)
print("运动任务环境已就绪；Jupyter 与运动任务可以使用不同 Python 环境。")


任务: basketball
实际推理模型: /home/ubuntu/workspaces/microduck_bpu_models/converted/basketball.opset11.onnx
运动任务环境已就绪；Jupyter 与运动任务可以使用不同 Python 环境。


In [3]:
def inspect_onnx(path):
    result = inspect_policy(path, POLICY_PYTHON)
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

contract = inspect_onnx(MODEL_PATH)
print("PASS-model-contract: 已检查当前任务实际使用的模型。")

{
  "inputs": [
    [
      "obs",
      [
        1,
        61
      ]
    ],
    [
      "h_in",
      [
        1,
        1,
        256
      ]
    ],
    [
      "c_in",
      [
        1,
        1,
        256
      ]
    ]
  ],
  "outputs": [
    [
      "actions",
      [
        1,
        14
      ]
    ],
    [
      "h_out",
      [
        1,
        1,
        256
      ]
    ],
    [
      "c_out",
      [
        1,
        1,
        256
      ]
    ]
  ],
  "nodes": 16,
  "metadata": {
    "run_path": "/scratch/basketball_continuations/b11_6999.pt",
    "joint_names": "left_hip_yaw,left_hip_roll,left_hip_pitch,left_knee,left_ankle,neck_pitch,head_pitch,head_yaw,head_roll,right_hip_yaw,right_hip_roll,right_hip_pitch,right_knee,right_ankle",
    "joint_stiffness": "1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000",
    "joint_damping": "0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000",
    "defaul

## 2. ONNX 导出入口

训练得到的是策略 checkpoint；导出时要把 actor 和观测归一化一起固化到 ONNX。下面是本任务命令模板。`--video` 会在本机录制仿真回放，新生成的视频写入 `outputs/`；本轮发布视频与运行报告保存在 `recordings/`。

```bash
uv run python scripts/export.py Mjlab-Basketball-MicroDuck\
  --checkpoint-file <CHECKPOINT.pt>\
  --onnx-file outputs/policy.onnx\
  --num-envs 1 --video --video-length 250
```

In [4]:
EXPORT_COMMAND = "python scripts/export_onnx.py run_dir=<FASTSAC_RUN_DIR> output=outputs/motrix_ball_balance_latest.onnx opset=11" if TASK_ID == "microduck-ball-balance" else ("uv run python scripts/export.py " + (TASK_ID or "<TASK_ID>") + " --checkpoint-file <CHECKPOINT.pt> --onnx-file outputs/policy.onnx --num-envs 1 --video --video-length 250")
print(EXPORT_COMMAND)
print("导出前后检查：观测维度、动作顺序、归一化和 action clip。")

uv run python scripts/export.py Mjlab-Basketball-MicroDuck --checkpoint-file <CHECKPOINT.pt> --onnx-file outputs/policy.onnx --num-envs 1 --video --video-length 250
导出前后检查：观测维度、动作顺序、归一化和 action clip。


## 3. 训练路径说明：从零或从预训练 checkpoint 开始

每个 Notebook 都把训练入口和回放入口分开：可以把 `resume=False` 作为从零初始化，也可以把 `resume=True`、任务专属 checkpoint 和对应的 actor/critic/优化器状态作为预训练续训。默认演示视频使用已经验证过的任务模型，不能把它描述成当前单元刚训练出来的模型。

这里的“10 个 iteration”是 10 轮采样加策略优化，不是每个动作只训练 10 步；例如 `64 个并行环境 × 24 个控制步 × 10` 才是约 `15360` 条 transition。导航 Notebook 明确不训练，直接接入已有模型；扰动和爬梯等 Notebook 也会标明它们使用的任务专属模型或官方双模型。

## 3. 官方篮球 b11 续训 smoke 与 ONNX 验证

篮球不能使用通用 PPO 入口，也不能从 walking checkpoint 续训。本单元严格调用官方
`scripts/finetune_basketball.py`：从 b11 checkpoint 恢复 actor、critic 和 Adam，固定学习率
`2e-5`，默认执行 `64 环境 × 5 iteration` smoke；训练成功后导出任务专属 ONNX，并运行
官方 PyTorch/ONNX parity 检查。smoke 输出写到 `outputs/basketball_official_smoke_latest.*`，
不会覆盖正式模型、正式 HBM 或发布目录。

默认不训练。确认 Ubuntu 已下载官方篮球包后，设置环境变量
`MICRODUCK_RUN_BASKETBALL_SMOKE=1` 再运行本单元。smoke 通过不等于模型可晋级；还必须用
同一评测协议比较原始 b11 与续训 checkpoint，稳定后才允许重新编译 HBM。

In [5]:
import shutil
import sys

OFFICIAL_ROOT = Path(os.getenv(
    "MICRODUCK_OFFICIAL_ROOT",
    "/home/ubuntu/workspaces/microduck-playground-official",
)).expanduser()
ARTIFACT_ROOT = Path(os.getenv(
    "MICRODUCK_BASKETBALL_ARTIFACT_ROOT",
    "/home/ubuntu/workspaces/microduck-basketball-artifacts",
)).expanduser()
SOURCE_CHECKPOINT = Path(os.getenv(
    "MICRODUCK_BASKETBALL_CHECKPOINT",
    str(ARTIFACT_ROOT / "checkpoint.pt"),
)).expanduser()
RUN_SMOKE = os.getenv("MICRODUCK_RUN_BASKETBALL_SMOKE", "0") == "1"
SMOKE_ENVS = int(os.getenv("MICRODUCK_BASKETBALL_SMOKE_ENVS", "64"))
SMOKE_ITERATIONS = int(os.getenv("MICRODUCK_BASKETBALL_SMOKE_ITERATIONS", "10"))
SMOKE_RUN_NAME = os.getenv(
    "MICRODUCK_BASKETBALL_SMOKE_RUN_NAME",
    "codex-basketball-smoke",
)
SMOKE_PT = OUTPUT_ROOT / "basketball_official_smoke_latest.pt"
SMOKE_ONNX = OUTPUT_ROOT / "basketball_official_smoke_latest.onnx"

print("官方 checkout:", OFFICIAL_ROOT)
print("官方 checkpoint:", SOURCE_CHECKPOINT)
print("smoke 输出:", SMOKE_PT, SMOKE_ONNX)

if not RUN_SMOKE:
    print("已跳过官方篮球 smoke：默认关闭。设置 MICRODUCK_RUN_BASKETBALL_SMOKE=1 后重新运行。")
elif not OFFICIAL_ROOT.is_dir():
    print("找不到官方 microduck-playground checkout，未覆盖任何输出。")
elif not SOURCE_CHECKPOINT.is_file():
    print("找不到官方 b11 checkpoint，未覆盖任何输出。")
else:
    train_cmd = [
        str(POLICY_PYTHON), "scripts/finetune_basketball.py", str(SOURCE_CHECKPOINT),
        "--run-name", SMOKE_RUN_NAME,
        "--num-envs", str(SMOKE_ENVS),
        "--iterations", str(SMOKE_ITERATIONS),
        "--learning-rate", "2e-5",
        "--push-interval-s", "1.5", "3",
        "--save-interval", str(SMOKE_ITERATIONS),
    ]
    train_env = os.environ.copy()
    train_env["WANDB_MODE"] = "offline"
    print("官方篮球 smoke:", " ".join(shlex.quote(x) for x in train_cmd))
    train_result = subprocess.run(train_cmd, cwd=OFFICIAL_ROOT, env=train_env, text=True)
    if train_result.returncode != 0:
        print("官方 smoke 失败，未覆盖输出。returncode:", train_result.returncode)
    else:
        run_root = OFFICIAL_ROOT / "logs" / "rsl_rl" / "basketball"
        smoke_runs = sorted(
            (p for p in run_root.glob(f"*_{SMOKE_RUN_NAME}") if p.is_dir()),
            key=lambda p: p.stat().st_mtime,
        )
        if not smoke_runs:
            raise FileNotFoundError(f"官方 smoke 成功但未找到输出目录: {run_root}")
        smoke_run = smoke_runs[-1]
        checkpoints = sorted(
            smoke_run.glob("model_*.pt"),
            key=lambda p: int(p.stem.split("_")[-1]),
        )
        if not checkpoints:
            raise FileNotFoundError(f"官方 smoke 输出没有 checkpoint: {smoke_run}")
        latest_checkpoint = checkpoints[-1]
        export_cmd = [
            str(POLICY_PYTHON), "scripts/export.py", "Mjlab-Basketball-MicroDuck",
            "--checkpoint-file", str(latest_checkpoint),
            "--onnx-file", str(SMOKE_ONNX),
        ]
        export_result = subprocess.run(export_cmd, cwd=OFFICIAL_ROOT, env=train_env, text=True)
        if export_result.returncode != 0 or not SMOKE_ONNX.is_file():
            raise RuntimeError("官方篮球 smoke ONNX 导出失败。")
        shutil.copy2(latest_checkpoint, SMOKE_PT)
        parity_cmd = [
            str(POLICY_PYTHON), "scripts/verify_basketball_onnx_parity.py",
            str(latest_checkpoint), str(SMOKE_ONNX),
        ]
        parity_result = subprocess.run(parity_cmd, cwd=OFFICIAL_ROOT, env=train_env, text=True)
        if parity_result.returncode != 0:
            raise RuntimeError("官方篮球 PyTorch/ONNX parity 检查失败。")
        CANDIDATE_MODEL_PATH = SMOKE_ONNX
        print('Training candidate:', CANDIDATE_MODEL_PATH, 'Stable inference model:', MODEL_PATH)
        print("PASS-official-basketball-smoke:", MODEL_PATH)
        print("实际 smoke checkpoint:", latest_checkpoint)

官方 checkout: /home/ubuntu/workspaces/microduck-playground-official
官方 checkpoint: /home/ubuntu/workspaces/microduck-basketball-artifacts/checkpoint.pt
smoke 输出: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/basketball_official_smoke_latest.pt /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/basketball_official_smoke_latest.onnx
已跳过官方篮球 smoke：默认关闭。设置 MICRODUCK_RUN_BASKETBALL_SMOKE=1 后重新运行。


## 4. 检查 ONNX 的输入与动作输出

先确认模型的输入名称、维度和动作输出，为接入仿真准备接口。下方使用零输入做一次模型检查。
随后在本任务闭环单元中读取实时观测，重新计算动作、下发执行器并更新物理状态。
第一次接触这条流程，请先运行 [MuJoCo 策略控制 HelloWorld](./00_MuJoCo策略控制_HelloWorld.ipynb)，
亲手执行前进、转向和停止，并查看 `data.ctrl` 与 `mujoco.mj_step()` 怎样连接策略和仿真。

In [6]:
local_check = local_model_check(MODEL_PATH, POLICY_PYTHON)
print(json.dumps(local_check, ensure_ascii=False, indent=2))
print("PASS-local-onnx: 输入输出检查完成，下一单元运行任务闭环。")

{
  "providers": [
    "CPUExecutionProvider"
  ],
  "inputs": [
    [
      "obs",
      [
        1,
        61
      ]
    ],
    [
      "h_in",
      [
        1,
        1,
        256
      ]
    ],
    [
      "c_in",
      [
        1,
        1,
        256
      ]
    ]
  ],
  "outputs": [
    [
      "actions",
      [
        1,
        14
      ]
    ],
    [
      "h_out",
      [
        1,
        1,
        256
      ]
    ],
    [
      "c_out",
      [
        1,
        1,
        256
      ]
    ]
  ],
  "finite": true
}
PASS-local-onnx: 输入输出检查完成，下一单元运行任务闭环。


## 5. 本任务策略闭环与 GPU 视频

运行下方单元会启动当前任务的策略与仿真器：每个控制周期读取机器人实时观测，
调用当前任务的 actor，取得关节动作，再交给仿真环境的 `env.step(actions)` 处理执行器与物理更新。
下一周期使用更新后的状态重新推理；视频记录这一段完整运行过程。
工作站完成物理计算、渲染与编码，GPU 执行当前任务的策略网络。

脚本位于 `02-可运行代码/microduck-playground-stilts/scripts/`。
行走、扰动、高跷、篮球和摆动使用 `gpu_inloop_video.py`；球平衡使用 `motrix_gpu_inloop_video.py`；
爬梯使用 `bpu_official_ladder_pair_video.py` 的 GPU 分支与爬梯/起身双策略。
下方模型路径需要指向当前任务的权重；各任务教程说明模型下载与环境准备。

若想逐行修改速度命令并直接看到关节目标下发，先运行 [HelloWorld](./00_MuJoCo策略控制_HelloWorld.ipynb)。
RDK 推理放在下一节，接收同一控制流程中的观测并返回动作。


本次模型检查与闭环使用同一个 `MODEL_PATH`。报告记录运行编号、真实调用次数和真实观测上的 GPU/ONNX 数值对照；缺少模型或推理失败会明确报错。

In [7]:
import sys

GPU_TASK = 'basketball'
GPU_VIDEO_STEPS = int(os.getenv("MICRODUCK_GPU_VIDEO_STEPS", "1500" if GPU_TASK == "basketball" else "900" if GPU_TASK == "ball_balance" else "500"))
GPU_VIDEO_OUTPUT = OUTPUT_ROOT / "gpu_videos" / f"{GPU_TASK}_gpu_latest.mp4" if GPU_TASK else None
GPU_ONNX_DEFAULTS = {
    "basketball": "/home/ubuntu/workspaces/microduck_bpu_models/converted/basketball.opset11.onnx",
    "walking": "/home/ubuntu/workspaces/microduck_bpu_models/converted/walking.opset11.onnx",
    "perturbation": "/home/ubuntu/workspaces/microduck_bpu_models/converted/walking.opset11.onnx",
    "stilt": "/home/ubuntu/workspaces/microduck_bpu_models/converted/stilts25.opset11.onnx",
    "swing": "/home/ubuntu/workspaces/microduck_bpu_models/converted/swing.opset11.onnx",
    "ball_balance": "/home/ubuntu/workspaces/microduck_bpu_models/converted/motrix_ball_balance_restored.opset11.onnx",
}
GPU_ONNX = MODEL_PATH
GPU_LADDER_CLIMBER = Path(os.getenv("MICRODUCK_GPU_LADDER_CLIMBER", "/home/ubuntu/workspaces/microduck_bpu_models/converted/official_ladder_climber.opset11.onnx")).expanduser()
GPU_LADDER_GETUP = Path(os.getenv("MICRODUCK_GPU_LADDER_GETUP", "/home/ubuntu/workspaces/microduck_bpu_models/converted/official_ladder_getup.opset11.onnx")).expanduser()
MOTRIX_ROOT = Path(os.getenv("MOTRIX_ROOT", "/home/ubuntu/workspaces/MotrixLab")).expanduser() if GPU_TASK == "ball_balance" else None
MOTRIX_PYTHON = Path(os.getenv("MOTRIX_PYTHON", str(MOTRIX_ROOT / ".venv/bin/python"))).expanduser() if MOTRIX_ROOT else None
MOTRIX_RUNNER = PLAYGROUND_ROOT / "scripts" / "motrix_runner.py" if GPU_TASK == "ball_balance" else None
GPU_SCRIPT = PLAYGROUND_ROOT / "scripts" / ("motrix_gpu_inloop_video.py" if GPU_TASK == "ball_balance" else "gpu_inloop_video.py")
GPU_LADDER_SCRIPT = PLAYGROUND_ROOT / "scripts" / "bpu_official_ladder_pair_video.py"

if GPU_TASK == "ladder":
    gpu_ready = GPU_LADDER_SCRIPT.is_file() and GPU_LADDER_CLIMBER.is_file() and GPU_LADDER_GETUP.is_file()
    gpu_cmd = [
        str(POLICY_PYTHON), str(GPU_LADDER_SCRIPT), "--backend", "gpu",
        "--official-root", "/home/ubuntu/workspaces/microduck-playground-official/experiments/desk-climb",
        "--climber-onnx", str(GPU_LADDER_CLIMBER), "--getup-onnx", str(GPU_LADDER_GETUP),
        "--output", str(GPU_VIDEO_OUTPUT), "--seconds", "24", "--num-envs", "64", "--record-env", "17",
    ]
else:
    gpu_ready = GPU_SCRIPT.is_file() and GPU_ONNX is not None and GPU_ONNX.is_file()
    gpu_cmd = [str(POLICY_PYTHON), str(GPU_SCRIPT)] if GPU_TASK != "ball_balance" else [str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "video", str(GPU_SCRIPT)]
    if GPU_TASK != "ball_balance":
        gpu_cmd += ["--task", GPU_TASK]
    gpu_cmd += ["--onnx", str(GPU_ONNX), "--output", str(GPU_VIDEO_OUTPUT), "--steps", str(GPU_VIDEO_STEPS)]

print("GPU 推理任务:", GPU_TASK)
print("GPU 模型:", GPU_ONNX if GPU_TASK != "ladder" else (GPU_LADDER_CLIMBER, GPU_LADDER_GETUP))
if not gpu_ready:
    raise FileNotFoundError("缺少任务模型、脚本或仿真环境；请按任务教程准备。")
elif GPU_TASK == "ball_balance" and (MOTRIX_PYTHON is None or not MOTRIX_PYTHON.is_file()):
    raise FileNotFoundError(MOTRIX_PYTHON)
else:
    GPU_VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    print("开始生成 GPU 推理视频。")
    import uuid
    GPU_RUN_ID = str(uuid.uuid4())
    gpu_env = os.environ.copy()
    gpu_env["MICRODUCK_VALIDATION_RUN_ID"] = GPU_RUN_ID
    gpu_result = subprocess.run(gpu_cmd, cwd=PLAYGROUND_ROOT, env=gpu_env, capture_output=True, text=True, timeout=600)
    safe_lines = [line for line in (gpu_result.stdout + "\n" + gpu_result.stderr).splitlines() if "GPU" in line or "PASS-" in line or "Video" in line]
    print("\n".join(safe_lines[-20:]))
    if gpu_result.returncode != 0 or not GPU_VIDEO_OUTPUT.is_file():
        log_path = OUTPUT_ROOT / "gpu_videos" / f"{GPU_TASK}_error.log"
        log_path.write_text(gpu_result.stdout + "\n" + gpu_result.stderr, encoding="utf-8")
        raise RuntimeError(f"GPU 推理失败；完整日志: {log_path}")
    GPU_REPORT = GPU_VIDEO_OUTPUT.with_suffix(".json")
    print("PASS-GPU-VIDEO:", GPU_VIDEO_OUTPUT)
    selected_models = [GPU_LADDER_CLIMBER, GPU_LADDER_GETUP] if GPU_TASK == "ladder" else [GPU_ONNX]
    gpu_report = verify_gpu_report(GPU_VIDEO_OUTPUT, GPU_RUN_ID, selected_models)
    print(gpu_report)
    print("PASS-FRESH-INFERENCE: 本次模型调用、关节动作与新视频已核对。")
    from IPython.display import Video, display
    display(Video(str(GPU_VIDEO_OUTPUT), embed=True))

GPU 推理任务: basketball
GPU 模型: /home/ubuntu/workspaces/microduck_bpu_models/converted/basketball.opset11.onnx
开始生成 GPU 推理视频。


GPU 策略推理已启用；不展示具体显卡型号。
PASS-GPU-VIDEO: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/gpu_videos/basketball_gpu_latest.mp4
PASS-GPU-REPORT: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/gpu_videos/basketball_gpu_latest.json
PASS-GPU-VIDEO: /home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/gpu_videos/basketball_gpu_latest.mp4
{'task': 'basketball', 'run_id': 'd569f5b0-c03f-4e5b-8b09-459cce110b9f', 'generated_at_utc': '2026-10-11T09:11:56.972060+00:00', 'policy_validation': {'inference_calls': 1500, 'real_observation_parity_checks': 32, 'max_abs_output_error': 7.152557373046875e-07, 'parity_atol': 0.0001, 'parity_rtol': 0.0001}, 'task_id': 'Mjlab-Basketball-MicroDuck', 'display_name': 'basketball-balance', 'backend': 'GPU ONNX via Torch CUDA', 'policy_device': 'GPU', 'onnx_file': '/home/ubuntu/w

![本次basketball GPU推理](../01-任务资料/精选视频/assets/notebook-replays/basketball_gpu.gif)

[播放 / 下载 MP4](recordings/gpu_videos/basketball_gpu_latest.mp4) · [运行报告](recordings/gpu_videos/basketball_gpu_latest.json)

MP4: recordings/gpu_videos/basketball_gpu_latest.mp4

## 6. RDK X5 BPU-in-the-loop 视频

本单元只展示新生成的板端闭环 MP4，不读取参考 GIF 或关键帧。Ubuntu 运行 MuJoCo 物理和 EGL 渲染；每一个控制步把任务观测发送到 RDK，RDK 用当前任务对应的 HBM 在 BPU 上计算 14 维动作，再返回给 Ubuntu。视频因此是“板端 BPU 产生动作、Ubuntu 负责仿真画面”的真实闭环。

运行前会自动重启 RDK 上的任务策略服务。切换 Notebook 时必须重新运行本单元，让板端 HBM 与当前任务匹配。

板端实践通过 `MICRODUCK_RUN_BPU=1` 启用；编译通过 `MICRODUCK_COMPILE_BPU=1` 启用。工作站推理可以独立运行，关闭板端时会输出 `NOT-RUN-BPU`。

In [8]:
RUN_BPU = os.getenv("MICRODUCK_RUN_BPU", "0") == "1"
if not RUN_BPU:
    print("NOT-RUN-BPU: 板端推理未启用；设置 MICRODUCK_RUN_BPU=1 后连接 RDK 实际运行。")
else:
    import hashlib
    import uuid
    BPU_RUN_ID = str(uuid.uuid4())
    bpu_env = os.environ.copy()
    bpu_env['MICRODUCK_VALIDATION_RUN_ID'] = BPU_RUN_ID
    import sys

    BPU_TASK = 'basketball'
    BPU_LADDER_LOCAL_OPT_IN = os.getenv("MICRODUCK_RUN_LOCAL_LADDER_BPU", "0") == "1"
    BPU_VIDEO_STEPS = int(os.getenv("MICRODUCK_BPU_VIDEO_STEPS", "1500" if BPU_TASK == "basketball" else "900" if BPU_TASK == "ball_balance" else "500"))
    BPU_VIDEO_WIDTH = int(os.getenv("MICRODUCK_BPU_VIDEO_WIDTH", "640"))
    BPU_VIDEO_HEIGHT = int(os.getenv("MICRODUCK_BPU_VIDEO_HEIGHT", "480"))
    BPU_CAMERA_DISTANCE = os.getenv("MICRODUCK_BPU_CAMERA_DISTANCE", "1.2")
    BPU_CAMERA_ELEVATION = os.getenv("MICRODUCK_BPU_CAMERA_ELEVATION", "-8")
    BPU_CAMERA_AZIMUTH = os.getenv("MICRODUCK_BPU_CAMERA_AZIMUTH", "90")
    BPU_PORT = int(os.getenv("MICRODUCK_BPU_PORT", "8765"))
    RDK_CONTROL_PORT = int(os.getenv("MICRODUCK_BPU_CONTROL_PORT", "8766"))
    RDK_HOST = os.getenv("RDK_HOST", "192.168.8.128")
    BPU_HBM_DEFAULTS = {
        "ball_balance": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/ball_balance_restored_int16_normalized_split_expanded/model_output/ball_balance_restored_int16_normalized_split_expanded.bin",
        "basketball": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/basketball_recalibrated_int16/model_output/basketball_recalibrated_int16.bin",
        "walking": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/walking/model_output/walking.opset11.bin",
        "perturbation": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/walking/model_output/walking.opset11.bin",
        "stilt": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/stilts25/model_output/stilts25.opset11.bin",
        "swing": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/swing/model_output/swing.opset11.bin",
        "ladder": "/home/ubuntu/workspaces/microduck_bpu_models/compiled/ladder/model_output/ladder.opset11.bin",
    }
    RDK_HBM_DEFAULTS = {
        "ball_balance": "/home/sunrise/microduck_policy/hbm/ball_balance.bin",
        "basketball": "/home/sunrise/microduck_policy/hbm/basketball.bin",
        "walking": "/home/sunrise/microduck_policy/hbm/walking.bin",
        "perturbation": "/home/sunrise/microduck_policy/hbm/walking.bin",
        "stilt": "/home/sunrise/microduck_policy/hbm/stilts25.bin",
        "swing": "/home/sunrise/microduck_policy/hbm/swing.bin",
        "ladder": "/home/sunrise/microduck_policy/hbm/ladder.bin",
    }
    BPU_HBM = Path(os.getenv("MICRODUCK_BPU_HBM", BPU_HBM_DEFAULTS.get(BPU_TASK, ""))) if BPU_TASK else None
    RDK_HBM = RDK_HBM_DEFAULTS.get(BPU_TASK, "") if BPU_TASK else ""
    RDK_SERVER_SCRIPT = os.getenv("RDK_BPU_SERVER_SCRIPT", "/home/sunrise/microduck_policy/rdk_bpu_policy_server.py")
    MOTRIX_ROOT = Path(os.getenv("MOTRIX_ROOT", "/home/ubuntu/workspaces/MotrixLab")).expanduser() if BPU_TASK == "ball_balance" else None
    MOTRIX_PYTHON = Path(os.getenv("MOTRIX_PYTHON", str(MOTRIX_ROOT / ".venv/bin/python"))).expanduser() if MOTRIX_ROOT else None
    MOTRIX_RUNNER = PLAYGROUND_ROOT / "scripts" / "motrix_runner.py" if BPU_TASK == "ball_balance" else None
    BPU_VIDEO_SCRIPT = PLAYGROUND_ROOT / "scripts" / ("motrix_bpu_inloop_video.py" if BPU_TASK == "ball_balance" else "bpu_inloop_video.py")
    BPU_VIDEO_OUTPUT = OUTPUT_ROOT / "bpu_videos" / f"{BPU_TASK}_bpu_latest.mp4" if BPU_TASK else None
    ALLOW_CACHED_BPU = False
    BPU_LADDER_PAIR_SCRIPT = PLAYGROUND_ROOT / "scripts" / "bpu_official_ladder_pair_video.py"
    BPU_LADDER_CLIMBER_HBM = Path(os.getenv("MICRODUCK_BPU_LADDER_CLIMBER_HBM", "/home/ubuntu/workspaces/microduck_bpu_models/compiled/official_ladder_climber_feature_normalized_all_int16/model_output/official_ladder_climber_feature_normalized_all_int16.bin")).expanduser()
    BPU_LADDER_GETUP_HBM = Path(os.getenv("MICRODUCK_BPU_LADDER_GETUP_HBM", "/home/ubuntu/workspaces/microduck_bpu_models/compiled/official_ladder_getup_feature_combined_all_int16/model_output/official_ladder_getup_feature_combined_all_int16.bin")).expanduser()
    BPU_LADDER_PAIR_OUTPUT = OUTPUT_ROOT / "bpu_videos" / "ladder_official_pair_bpu_latest.mp4"

    def show_cached_bpu_video(path, reason):
        if not ALLOW_CACHED_BPU or path is None or not path.is_file():
            return False
        print("BPU-CACHED-VIDEO:", path)
        print("说明:", reason)
        report = path.with_suffix(".json")
        if report.is_file():
            print(json.loads(report.read_text(encoding="utf-8")))
        from IPython.display import Video, display
        display(Video(str(path), embed=True))
        return True

    def restart_rdk_policy_server():
        global BPU_BOARD_SHA
        if not BPU_TASK or not RDK_HBM:
            return False
        try:
            if BPU_HBM is not None and BPU_HBM.is_file():
                payload = BPU_HBM.read_bytes()
                request = {
                    "op": "upload_and_switch",
                    "task": BPU_TASK,
                    "size": len(payload),
                    "sha256": hashlib.sha256(payload).hexdigest(),
                }
            else:
                payload = b""
                request = {"op": "switch", "task": BPU_TASK}
            transform_path = BPU_HBM.with_suffix(".input_transform.json")
            if payload and transform_path.is_file():
                request["input_transform"] = json.loads(transform_path.read_text(encoding="utf-8"))
            with socket.create_connection((RDK_HOST, RDK_CONTROL_PORT), timeout=8) as control:
                control.sendall((json.dumps(request) + "\n").encode("utf-8"))
                if payload:
                    for offset in range(0, len(payload), 1024 * 1024):
                        control.sendall(payload[offset:offset + 1024 * 1024])
                response = json.loads(control.makefile("rb").readline().decode("utf-8"))
            if response.get("ok"):
                BPU_BOARD_SHA = response.get("sha256")
                if BPU_BOARD_SHA != hashlib.sha256(BPU_HBM.read_bytes()).hexdigest():
                    raise ValueError("板端模型 SHA 与所选 HBM 不一致。")
                action = "已上传并切换" if payload else "已切换"
                print("RDK 任务策略服务" + action + ":", BPU_TASK, "->", response.get("model"), "pid=", response.get("pid"))
                return True
            print("RDK BPU 控制服务拒绝切换:", response)
        except (OSError, ValueError, json.JSONDecodeError) as exc:
            print("RDK BPU 控制服务不可用，尝试 SSH:", exc)
        remote = "sunrise@" + RDK_HOST
        command = (
            "pkill -f '^python3 .*rdk_bpu_policy_server.py' >/dev/null 2>&1 || true; "
            "sleep 1; "
            "nohup python3 " + shlex.quote(RDK_SERVER_SCRIPT) +
            " --model " + shlex.quote(RDK_HBM) +
            " --port " + str(BPU_PORT) +
            " >/tmp/microduck_bpu_server.log 2>&1 </dev/null &"
        )
        result = subprocess.run(
            ["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=8", remote, "bash -lc " + shlex.quote(command)],
            capture_output=True, text=True,
        )
        if result.returncode:
            print("RDK 策略服务启动失败:", result.stderr.strip() or result.stdout.strip())
            return False
        print("RDK 任务策略服务已切换:", BPU_TASK, "->", RDK_HBM)
        return True

    if BPU_TASK == "ladder":
        pair_ready = BPU_LADDER_PAIR_SCRIPT.is_file() and BPU_LADDER_CLIMBER_HBM.is_file() and BPU_LADDER_GETUP_HBM.is_file()
        with socket.create_connection((RDK_HOST, RDK_CONTROL_PORT), timeout=8) as control:
            control.sendall(b'{"op":"switch_pair"}\n')
            pair_status = json.loads(control.makefile("rb").readline())
        if not pair_status.get("ok"):
            raise RuntimeError(pair_status)
        pair_hashes = [pair_status["models"][role]["sha256"] for role in ("climber", "getup")]
        pair_cmd = [
            str(POLICY_PYTHON), str(BPU_LADDER_PAIR_SCRIPT), "--backend", "bpu",
            "--climber-hbm", str(BPU_LADDER_CLIMBER_HBM), "--getup-hbm", str(BPU_LADDER_GETUP_HBM),
            "--output", str(BPU_LADDER_PAIR_OUTPUT), "--seconds", os.getenv("MICRODUCK_LADDER_SECONDS", "24"),
            "--num-envs", "64", "--record-env", "17",
        ]
        print("BPU 官方双模型:", BPU_LADDER_CLIMBER_HBM, BPU_LADDER_GETUP_HBM)
        if not pair_ready:
            raise FileNotFoundError("缺少 climber/get-up HBM 或脚本。")
        else:
            BPU_LADDER_PAIR_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
            pair_result = subprocess.run(pair_cmd, cwd=PLAYGROUND_ROOT, env=bpu_env, capture_output=True, text=True, timeout=600)
            print("\n".join((pair_result.stdout + "\n" + pair_result.stderr).splitlines()[-35:]))
            if pair_result.returncode != 0 or not BPU_LADDER_PAIR_OUTPUT.is_file():
                if not show_cached_bpu_video(BPU_LADDER_PAIR_OUTPUT, "本轮板端不可达，保留此前真实 RDK X5 BPU 双模型视频。"):
                    raise RuntimeError("官方爬梯 BPU 双模型视频生成失败。")
            else:
                print("PASS-BPU-VIDEO:", BPU_LADDER_PAIR_OUTPUT)
                pair_report = BPU_LADDER_PAIR_OUTPUT.with_suffix(".json")
                if pair_report.is_file():
                    print(verify_bpu_report(BPU_LADDER_PAIR_OUTPUT, BPU_RUN_ID, [BPU_LADDER_CLIMBER_HBM, BPU_LADDER_GETUP_HBM], pair_hashes))
                    print("PASS-FRESH-BPU: 本次双模型板端动作已驱动仿真并生成视频。")
                from IPython.display import Video, display
                display(Video(str(BPU_LADDER_PAIR_OUTPUT), embed=True))
    elif not BPU_TASK:
        print("该任务目前没有可验证的任务专属 HBM；不会用其他任务视频替代。")
    elif BPU_TASK == "ladder" and not BPU_LADDER_LOCAL_OPT_IN:
        print("已跳过本地 14 级 ladder HBM 视频：它不是社区 climber/get-up 方案。")
        print("只有明确设置 MICRODUCK_RUN_LOCAL_LADDER_BPU=1 才会运行本地实验，不得将结果标为社区复刻。")
    elif BPU_HBM is None or not BPU_HBM.is_file():
        raise FileNotFoundError(BPU_HBM)
    elif not BPU_VIDEO_SCRIPT.is_file():
        raise FileNotFoundError(BPU_VIDEO_SCRIPT)
    elif BPU_TASK == "ball_balance" and (MOTRIX_PYTHON is None or not MOTRIX_PYTHON.is_file()):
        print("找不到 MotrixLab Python 环境:", MOTRIX_PYTHON)
    elif BPU_TASK == "ball_balance" and (MOTRIX_RUNNER is None or not MOTRIX_RUNNER.is_file()):
        print("找不到 MotrixLab 启动器:", MOTRIX_RUNNER)
    elif restart_rdk_policy_server():
        BPU_VIDEO_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
        if BPU_TASK == "ball_balance":
            video_cmd = [str(MOTRIX_PYTHON), str(MOTRIX_RUNNER), "video", str(BPU_VIDEO_SCRIPT)]
        else:
            video_cmd = [str(POLICY_PYTHON), str(BPU_VIDEO_SCRIPT)]
        if BPU_TASK != "ball_balance":
            video_cmd += ["--task", BPU_TASK]
        video_cmd += [
            "--bpu-host", RDK_HOST, "--bpu-port", str(BPU_PORT),
            "--hbm", str(BPU_HBM), "--output", str(BPU_VIDEO_OUTPUT),
            "--steps", str(BPU_VIDEO_STEPS), "--width", str(BPU_VIDEO_WIDTH),
            "--height", str(BPU_VIDEO_HEIGHT),
        ]
        video_cmd += [
            "--camera-distance", BPU_CAMERA_DISTANCE,
            "--camera-elevation", BPU_CAMERA_ELEVATION,
            "--camera-azimuth", BPU_CAMERA_AZIMUTH,
        ]
        print("开始生成板端闭环视频:", " ".join(shlex.quote(x) for x in video_cmd))
        video_result = subprocess.run(
            video_cmd, cwd=PLAYGROUND_ROOT, env=bpu_env,
            capture_output=True, text=True,
        )
        print("\n".join((video_result.stdout + "\n" + video_result.stderr).splitlines()[-35:]))
        if video_result.returncode != 0 or not BPU_VIDEO_OUTPUT.is_file():
            raise RuntimeError("BPU-in-the-loop 视频生成失败，请先查看上面最后 35 行日志。")
        BPU_REPORT = BPU_VIDEO_OUTPUT.with_suffix(".json")
        print("PASS-BPU-VIDEO:", BPU_VIDEO_OUTPUT)
        if BPU_REPORT.is_file():
            print(verify_bpu_report(BPU_VIDEO_OUTPUT, BPU_RUN_ID, [BPU_HBM], [BPU_BOARD_SHA]))
            print("PASS-FRESH-BPU: 本次板端动作已驱动仿真并生成视频。")
        from IPython.display import Video, display
        display(Video(str(BPU_VIDEO_OUTPUT), embed=True))
    else:
        if not show_cached_bpu_video(BPU_VIDEO_OUTPUT, "板端不可达，展示历史视频。"):
            raise ConnectionError("RDK 板端推理未完成；请检查开发板网络与策略服务。")



RDK 任务策略服务已上传并切换: basketball -> /home/sunrise/microduck_policy/hbm/basketball.bin pid= 44100
开始生成板端闭环视频: /home/ubuntu/workspaces/microduck_rl/.venv/bin/python '/home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/02-可运行代码/microduck-playground-stilts/scripts/bpu_inloop_video.py' --task basketball --bpu-host 192.168.8.128 --bpu-port 8765 --hbm /home/ubuntu/workspaces/microduck_bpu_models/compiled/basketball_recalibrated_int16/model_output/basketball_recalibrated_int16.bin --output '/home/ubuntu/workspaces/microduck-teaching/microduck-topic/16-专题组队学习/05-MicroDuck运动策略与任务开发专题/03-Notebook/outputs/bpu_videos/basketball_bpu_latest.mp4' --steps 1500 --width 640 --height 480 --camera-distance 1.2 --camera-elevation -8 --camera-azimuth 90



[INFO] <CurriculumManager> contains 4 active terms.
+----------------------------+
|  Active Curriculum Terms   |
+-------+--------------------+
| Index | Name               |
+-------+--------------------+
|   0   | action_rate_weight |
|   1   | standing_envs      |
|   2   | com_range          |
|   3   | head_com_range     |
+-------+--------------------+

[INFO] <MetricsManager> contains 1 active terms.
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

[INFO] <NullRecorderManager> (inactive)
Module mujoco_warp._src.io 4b387a0 load on device 'cuda:0' took 1.40 ms  (cached)
Module _tile_cholesky_factorize__locals__cholesky_factorize_68698e2e e5a5250 load on device 'cuda:0' took 14.29 ms  (cached)
Module _tile_cholesky_factorize__locals__cholesky_factorize_a02b7349 3d3b894 load on device 'cuda:0' took 14.00 ms  (cached)
Module _tile_chole

![本次basketball BPU推理](../01-任务资料/精选视频/assets/notebook-replays/basketball_bpu.gif)

[播放 / 下载 MP4](recordings/bpu_videos/basketball_bpu_latest.mp4) · [运行报告](recordings/bpu_videos/basketball_bpu_latest.json)

MP4: recordings/bpu_videos/basketball_bpu_latest.mp4

## 7. RDK X5 / BPU 审计

上一单元已经生成任务专属闭环视频。本单元只检查板端连通、HBM 元数据和服务日志；通用 MobileNet 样例只能证明系统运行时存在，不能代替任务视频。

板端实践通过 `MICRODUCK_RUN_BPU=1` 启用；编译通过 `MICRODUCK_COMPILE_BPU=1` 启用。工作站推理可以独立运行，关闭板端时会输出 `NOT-RUN-BPU`。

In [9]:
RUN_BPU = os.getenv("MICRODUCK_RUN_BPU", "0") == "1"
if not RUN_BPU:
    print("NOT-RUN-BPU: 本轮只运行工作站推理。")
else:
    print("板端推理结果:")
    if BPU_TASK == "ladder":
        board_report = verify_bpu_report(BPU_LADDER_PAIR_OUTPUT, BPU_RUN_ID,
            [BPU_LADDER_CLIMBER_HBM, BPU_LADDER_GETUP_HBM], pair_hashes)
    else:
        board_report = verify_bpu_report(BPU_VIDEO_OUTPUT, BPU_RUN_ID, [BPU_HBM], [BPU_BOARD_SHA])
    print("PASS-FRESH-BPU: 模型 SHA、运行编号与实际控制步数已核对。")
    print("控制步数:", board_report["steps"])
    print("重置次数:", board_report.get("resets", board_report.get("terminated_or_truncated_resets", 0)))
    print("视频由本次任务策略驱动物理仿真生成。")

板端推理结果:
PASS-FRESH-BPU: 模型 SHA、运行编号与实际控制步数已核对。
控制步数: 1500
重置次数: 0
视频由本次任务策略驱动物理仿真生成。


## 8. 阅读本次运行结果

先查看模型接口，再查看任务闭环与生成的视频，最后对照板端报告。

| 输出标记 | 对应流程 |
| --- | --- |
| `PASS-local-onnx` | 模型结构、输入输出形状和有限值检查通过；随后进入任务闭环 |
| `PASS-GPU-VIDEO` | GPU 策略连接物理仿真，完成本次视频与报告生成 |
| `PASS-BPU-VIDEO` | 当前连接 RDK，逐步取得对应 HBM 的动作，驱动工作站仿真并生成视频 |
| `BPU-CACHED-VIDEO` | 播放之前保存的板端演示；连接 RDK 后重跑板端闭环单元可生成新视频 |
| `PASS-rdk-bpu-runtime-sample` | 开发板的 MobileNet 样例完成推理，运行时检查通过 |
| `PASS-rdk-bpu` | 当前任务 HBM 的板端推理完成，输入输出与耗时保存在输出中 |

任务效果结合位移、速度、稳定时间、接触或登桌指标与视频一起阅读。
修改模型、命令或场景后重新运行对应闭环单元，保存自己的实验记录。

板端实践通过 `MICRODUCK_RUN_BPU=1` 启用；编译通过 `MICRODUCK_COMPILE_BPU=1` 启用。工作站推理可以独立运行，关闭板端时会输出 `NOT-RUN-BPU`。